In [1]:
import numpy as np
import scanpy as sc
import spapros as sp
import pandas as pd
import anndata as ad
import os

# ============================================================
# 1. LOAD + SUBSET + NORMALIZATION
# ============================================================

adata_full = ad.read_h5ad(
    "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/abc_atlas.h5ad",
    backed='r'
)

np.random.seed(42)
idx = np.sort(np.random.choice(adata_full.n_obs, size=200000, replace=False))
adata = adata_full[idx, :].to_memory()

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

print("Loaded and normalized:", adata)


# ============================================================
# 2. BUILD GENOMIC REGIONS FOR PROBE DESIGN
#    This extracts coding exons + exon junctions
# ============================================================

region_generator = sp.pr.GenomicRegionGenerator(
    transcript_source="Ensembl",
    species="mouse",              # <-- IMPORTANT (set to human if needed)
    fasta_source="Ensembl",
    min_region_len=300
)

regions = region_generator.create_transcript_regions(
    gene_ids=adata.var_names.tolist(),
    regions_folder="probe_design/regions",
    return_regions=True
)

print(f"Regions generated for {len(regions)} genes.")


# ============================================================
# 3. DESIGN RAW PROBES FOR MERFISH
# ============================================================

probe_designer = sp.pr.MerfishProbeDesigner(
    probe_len=30,
    gc_min=0.3, gc_max=0.7,
    tm_min=60, tm_max=80,
    homopolymer=5,
    n_probes_per_region=10,     # initial raw probes per region
    save_dir="probe_design/probes_raw"
)

probe_designer.create_probe_database(genomic_regions=regions)

print("Raw probes generated.")


# ============================================================
# 4. FILTER PROBES BY GC/Tm/Homopolymers
# ============================================================

probe_designer.filter_by_property(
    filter_folder="probe_design/probes_filtered"
)

print("Property filtering done.")


# ============================================================
# 5. FILTER BY SEQUENCE SPECIFICITY USING BLAST
# ============================================================

probe_designer.filter_by_specificity(
    filter_folder="probe_design/probes_specific",
    match_fraction=0.2,         # typical value
    match_score=80              # strong filter — adjust if needed
)

print("BLAST specificity filtering done.")


# ============================================================
# 6. ASSEMBLE FINAL PROBESETS PER GENE
# ============================================================

probe_sets = probe_designer.create_probe_sets(
    probeset_folder="probe_design/probesets",
    probeset_size_min=3,       # MERFISH usually needs ≥3
    probeset_size_opt=5
)

print(f"Probe sets assembled for {len(probe_sets)} genes.")


# ============================================================
# 7. SUMMARIZE GENE FEASIBILITY
# ============================================================

summary = []

for gene_id in adata.var_names:
    if gene_id in probe_sets:
        n_probes = len(probe_sets[gene_id])
        score = min(1.0, n_probes / 5)   # simple feasibility score (0–1)
        ok = n_probes >= 3
    else:
        n_probes = 0
        score = 0.0
        ok = False

    summary.append({
        "gene_id": gene_id,
        "n_valid_probes": n_probes,
        "feasibility_score": score,
        "passes_merfish_constraints": ok
    })

df = pd.DataFrame(summary)


# ============================================================
# 8. SAVE RESULTS
# ============================================================

df.to_csv("outputs/merfish_gene_feasibility.csv", index=False)

df[df["passes_merfish_constraints"]].to_csv(
    "outputs/merfish_feasible_genes_only.csv",
    index=False
)

print("Saved:")
print(" - outputs/merfish_gene_feasibility.csv")
print(" - outputs/merfish_feasible_genes_only.csv")



/bin/sh: line 1: blastn: command not found
Loaded and normalized: AnnData object with n_obs × n_vars = 200000 × 32285
    obs: 'abc_sample_id', 'anatomical_division_label', 'barcoded_cell_sample_label', 'brain_section_label', 'cell_barcode', 'class', 'class_color', 'cluster', 'cluster_alias', 'cluster_color', 'dataset_label', 'donor_genotype', 'donor_label', 'donor_sex', 'entity', 'feature_matrix_label', 'library_label', 'library_method', 'neurotransmitter', 'neurotransmitter_color', 'region_of_interest_acronym', 'region_of_interest_color', 'region_of_interest_order', 'subclass', 'subclass_color', 'supertype', 'supertype_color', 'x', 'y'
    uns: 'log1p'


AttributeError: module 'spapros' has no attribute 'pr'

In [2]:
import spapros as sp
print("Spapros version:", sp.__version__)

from spapros.design import GenomicRegionGenerator

print("Probe-design module successfully imported!")



Spapros version: 0.1.6


ModuleNotFoundError: No module named 'spapros.design'